Context-Aware Quantum-Inspired Intelligence for AI-Aided Cyber Worm Detection
Simulation with LSTM, SHAP, and LIME

In [ ]:
# -*- coding: utf-8 -*-
"""
Context-Aware Quantum-Inspired Intelligence for AI-Aided Cyber Worm Detection
Simulation with LSTM, SHAP, and LIME

This Jupyter Notebook provides a simulation using an LSTM model for predicting
different network traffic types, leveraging a custom dataset from a CSV file.
It incorporates context-aware quantum-inspired intelligence by performing
feature selection on network and contextual features and analyzes performance
with and without context-aware threat analysis.  It also adds
SHAP and LIME explainability to the LSTM model's predictions.

The notebook includes:
    1. Loading and preprocessing a custom dataset, separating network and contextual features.
    2. Preparing sequential data for the LSTM model.
    3. Building and training the LSTM model for multiclass classification.
    4. Evaluating the model with accuracy, precision, recall, F1-score, and loss.
    5. Plotting accuracy and loss curves during training.
    6. Generating and visualizing the confusion matrix for both scenarios.
    7. Plotting the AUC-ROC curve for each traffic type (One-vs-Rest).
    8. Using SHAP and LIME to explain LSTM predictions.
    9. Context-aware threat analysis and performance comparison.

Assumptions about the CSV file:
    - It contains network traffic features and contextual features.
    - One column is the 'traffic_type' column with more than 2 unique traffic types.
    - Data is structured or can be structured as sequential data.
    - Non-numeric columns are converted to numeric.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import random

# Explainability
import shap
import lime
import lime.lime_tabular

# Seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

# --- 1. Load and Preprocess Custom Dataset ---
csv_file_path = 'network_traffic_data.csv'  # Replace with your CSV file path

try:
    df = pd.read_csv(csv_file_path)
    print(f"Dataset loaded successfully from {csv_file_path}")
except FileNotFoundError:
    print(f"Error: CSV file not found at {csv_file_path}")
    exit()

target_column = 'traffic_type'
if target_column not in df.columns:
    print(f"Error: Target column '{target_column}' not found.")
    exit()

y_original = df[target_column].values
X = df.drop(columns=[target_column])

# Convert non-numeric columns to numeric
for column in X.columns:
    if pd.api.types.is_object_dtype(X[column]):
        try:
            X[column] = pd.to_numeric(X[column], errors='raise')
        except ValueError:
            print(f"Converting non-numeric column: {column}")
            le = LabelEncoder()
            X[column] = le.fit_transform(X[column])
    elif not pd.api.types.is_numeric_dtype(X[column]):
        print(f"Warning: Column '{column}' is not numeric and not string. Handling needed.")

# --- 2. Feature Selection Preparation ---
# Separate network and contextual features.
network_feature_names = [
    'src_ip', 'src_port', 'dst_ip', 'dst_port', 'protocol', 'packet_size',
    'flow_duration', 'bytes_sent', 'bytes_received', 'packets_sent',
    'packets_received'  # Example network feature names
]
contextual_feature_names = [
    'timestamp', 'user_id', 'application_name', 'location', 'time_of_day',
    'day_of_week'  # Example contextual feature names
]

# Ensure the feature names exist in the DataFrame
network_features = [col for col in network_feature_names if col in X.columns]
contextual_features = [col for col in contextual_feature_names if col in X.columns]

# Create separate DataFrames for network and contextual features
X_network = X[network_features].values
X_contextual = X[contextual_features].values

unique_traffic_types = np.unique(y_original)
num_classes = len(unique_traffic_types)
print(f"Number of unique traffic types: {num_classes}")

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_original)
y_categorical = to_categorical(y_encoded, num_classes=num_classes)

# --- 3. Simplified Quantum-Inspired Particle Swarm Optimization (QPSO) ---
def simple_fitness_evaluation(feature_indices, data, labels):
    """
    Simplified fitness function (variance ratio).
    """
    if not feature_indices:
        return 0
    normal_data = data[labels == 0][:, feature_indices]
    anomalous_data = data[labels == 1][:, feature_indices]
    if normal_data.size == 0 or anomalous_data.size == 0:
        return 0
    return np.var(anomalous_data) / (np.var(normal_data) + 1e-6)

def qpso(data, labels, num_particles=10, iterations=20, num_features_to_select=10):
    """
    Simplified QPSO implementation.
    """
    n_features = data.shape[1]
    particles = [np.random.choice(n_features, num_features_to_select, replace=False) for _ in range(num_particles)]
    p_best_positions = list(particles)
    p_best_fitnesses = [simple_fitness_evaluation(p, data, labels) for p in particles]
    g_best_position = particles[np.argmax(p_best_fitnesses)]
    g_best_fitness = max(p_best_fitnesses)

    for _ in range(iterations):
        for i in range(num_particles):
            # Simplified particle update
            phi = np.random.rand()
            p = particles[i]
            x_mean = np.mean(np.array([p_best_positions[i], g_best_position]), axis=0).astype(int)
            new_position = np.zeros_like(p)
            for j in range(len(p)):
                if np.random.rand() < phi:
                    new_position[j] = x_mean[j]
                else:
                    new_position[j] = p[j]

            # Ensure no duplicate features in a particle
            new_position = np.unique(new_position)
            if len(new_position) != num_features_to_select:
                remaining_features = set(range(n_features)) - set(new_position)
                new_position = np.concatenate((new_position, np.random.choice(list(remaining_features), num_features_to_select - len(new_position), replace=False)))
            particles[i] = new_position

            # Evaluate fitness
            current_fitness = simple_fitness_evaluation(particles[i], data, labels)

            # Update p_best
            if current_fitness > p_best_fitnesses[i]:
                p_best_fitnesses[i] = current_fitness
                p_best_positions[i] = particles[i]

            # Update g_best
            if current_fitness > g_best_fitness:
                g_best_fitness = current_fitness
                g_best_position = particles[i]

    return g_best_position

# --- 4. Feature Selection with QPSO on Network and Contextual Features ---
anomaly_labels = (y_original == 'Worm').astype(int)  # Example: 'Worm' is the anomaly

num_network_features_to_select = 5  # Tune this parameter
num_contextual_features_to_select = 5  # Tune this parameter

print("\n--- QPSO Feature Selection for Network Features ---")
selected_network_feature_indices = qpso(
    X_network, anomaly_labels, num_features_to_select=num_network_features_to_select
)
print(f"Selected network feature indices: {selected_network_feature_indices}")

print("\n--- QPSO Feature Selection for Contextual Features ---")
selected_contextual_feature_indices = qpso(
    X_contextual, anomaly_labels, num_features_to_select=num_contextual_features_to_select
)
print(f"Selected contextual feature indices: {selected_contextual_feature_indices}")

# Reduce the feature sets
X_network_selected = X_network[:, selected_network_feature_indices]
X_contextual_selected = X_contextual[:, selected_contextual_feature_indices]

# Combine the selected features
X_selected_context_aware = np.concatenate((X_network_selected, X_contextual_selected), axis=1)

# --- 5. Prepare Data for LSTM (Context-Aware) ---
n_samples_context_aware = X_selected_context_aware.shape[0]
n_features_context_aware = X_selected_context_aware.shape[1]
n_time_steps = 10

def create_sequences(data, labels, time_steps):
    Xs, ys = [], []
    for i in range(len(data) - time_steps):
        Xs.append(data[i:(i + time_steps)])
        ys.append(labels[i + time_steps - 1])
    return np.array(Xs), np.array(ys)

X_sequences_context_aware, y_sequences_context_aware = create_sequences(X_selected_context_aware, y_categorical, n_time_steps)

# Split data
X_train_context_aware, X_test_context_aware, y_train_context_aware, y_test_context_aware, y_train_original_context_aware, y_test_original_context_aware = train_test_split(
    X_sequences_context_aware, y_sequences_context_aware, y_original[n_time_steps-1:], test_size=0.2, random_state=42, stratify=y_original[n_time_steps-1:]
)

# Scale the data
scaler_context_aware = StandardScaler()
X_train_scaled_context_aware = scaler_context_aware.fit_transform(X_train_context_aware.reshape(-1, n_features_context_aware)).reshape(X_train_context_aware.shape)
X_test_scaled_context_aware = scaler_context_aware.transform(X_test_context_aware.reshape(-1, n_features_context_aware)).reshape(X_test_context_aware.shape)

# --- 6. Build and Train the LSTM Model (Context-Aware) ---
model_context_aware = Sequential()
model_context_aware.add(LSTM(64, activation='relu', input_shape=(n_time_steps, n_features_context_aware)))
model_context_aware.add(Dense(num_classes, activation='softmax'))

optimizer_context_aware = tf.keras.optimizers.SGD(learning_rate=0.01)
model_context_aware.compile(optimizer=optimizer_context_aware, loss='categorical_crossentropy', metrics=['accuracy'])
model_context_aware.summary()

epochs = 50
batch_size = 32

history_context_aware = model_context_aware.fit(X_train_scaled_context_aware, y_train_context_aware, epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose=1)

# --- 7. Evaluate the Model and Get Metrics (Context-Aware) ---
y_pred_prob_context_aware = model_context_aware.predict(X_test_scaled_context_aware)
y_pred_classes_context_aware = np.argmax(y_pred_prob_context_aware, axis=1)
y_pred_original_context_aware = label_encoder.inverse_transform(y_pred_classes_context_aware)

print("\n--- Evaluation Metrics (Context-Aware) ---")
accuracy_context_aware = accuracy_score(y_test_original_context_aware, y_pred_original_context_aware)
precision_context_aware = precision_score(y_test_original_context_aware, y_pred_original_context_aware, average='weighted')
recall_context_aware = recall_score(y_test_original_context_aware, y_pred_original_context_aware, average='weighted')
f1_context_aware = f1_score(y_test_original_context_aware, y_pred_original_context_aware, average='weighted')
loss_context_aware = model_context_aware.evaluate(X_test_scaled_context_aware, y_test_context_aware, verbose=0)[0]

print(f"Accuracy: {accuracy_context_aware:.4f}")
print(f"Precision (weighted): {precision_context_aware:.4f}")
print(f"Recall (weighted): {recall_context_aware:.4f}")
print(f"F1-Score (weighted): {f1_context_aware:.4f}")
print(f"Loss: {loss_context_aware:.4f}")

print("\n--- Classification Report (Context-Aware) ---")
print(classification_report(y_test_original_context_aware, y_pred_original_context_aware))

# --- 8. Plotting Training History (Context-Aware) ---
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_context_aware.history['accuracy'], label='Train Accuracy')
plt.plot(history_context_aware.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy (Context-Aware)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_context_aware.history['loss'], label='Train Loss')
plt.plot(history_context_aware.history['val_loss'], label='Validation Loss')
plt.title('Model Loss (Context-Aware)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

# --- 9. Confusion Matrix (Context-Aware) ---
conf_matrix_context_aware = confusion_matrix(y_test_original_context_aware, y_pred_original_context_aware)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_context_aware, annot=True, fmt='d', cmap='Blues',
            xticklabels=unique_traffic_types, yticklabels=unique_traffic_types)
plt.xlabel('Predicted Traffic Type')
plt.ylabel('Actual Traffic Type')
plt.title('Confusion Matrix (LSTM - Context-Aware)')
plt.show()

# --- 10. AUC-ROC Curve (Context-Aware) ---
plt.figure(figsize=(10, 8))
lw = 2

y_test_encoded_context_aware = label_encoder.transform(y_test_original_context_aware)
y_test_onehot_context_aware = to_categorical(y_test_encoded_context_aware, num_classes=num_classes)

for i, traffic_type in enumerate(unique_traffic_types):
    fpr_context_aware, tpr_context_aware, _ = roc_curve(y_test_onehot_context_aware[:, i], y_pred_prob_context_aware[:, i])
    roc_auc_context_aware = auc(fpr_context_aware, tpr_context_aware)
    plt.plot(fpr_context_aware, tpr_context_aware, lw=lw, label=f'ROC curve (type {traffic_type}, AUC = {roc_auc_context_aware:.2f})')

plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) - One-vs-Rest (Context-Aware)')
plt.legend(loc="lower right")
plt.show()

# --- 11. Explainability (SHAP and LIME) - Context Aware ---
# SHAP - Global Explainability
print("\n--- SHAP Analysis (Context-Aware) ---")
def model_predict_for_shap_context_aware(X):
    """
    Wrapper function for SHAP to handle time series input.
    """
    if len(X.shape) == 2:
        X = np.expand_dims(X, axis=1)
    return model_context_aware.predict(X)

background_context_aware = X_train_scaled_context_aware[np.random.choice(X_train_scaled_context_aware.shape[0], 100, replace=False)]
explainer_context_aware = shap.DeepExplainer(model_context_aware, background_context_aware)

shap_values_context_aware = explainer_context_aware.shap_values(X_test_scaled_context_aware)

print("SHAP values shape (Context-Aware):", np.array(shap_values_context_aware).shape)
if len(shap_values_context_aware) > 0:
    shap.summary_plot(shap_values_context_aware, X_test_scaled_context_aware, feature_names=[f'Feature_{i}' for i in range(n_features_context_aware)])

# LIME - Local Explainability
print("\n--- LIME Analysis (Context-Aware) ---")
explainer_lime_context_aware = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled_context_aware.reshape(X_train_scaled_context_aware.shape[0], n_time_steps * n_features_context_aware),
    feature_names=[f'Feature_{i}' for i in range(n_time_steps * n_features_context_aware)],
    class_names=unique_traffic_types,
    discretize_continuous=True
)

explanation_context_aware = explainer_lime_context_aware.explain_instance(
    data_row=X_test_scaled_context_aware[0].reshape(-1),
    predict_fn=lambda x: model_context_aware.predict(x.reshape(-1, n_time_steps, n_features_context_aware)),
    top_labels=num_classes
)
print(explanation_context_aware.as_text())
explanation_context_aware.show_in_notebook(show_table=True, show_all=True)


# --- 12. Performance Comparison with No Context-Awareness ---
# To compare, we need a model trained *without* the contextual features.
# So, we repeat the LSTM training process, but this time using only X_network.

# --- 12.1 Prepare Data for LSTM (No Context-Awareness) ---
X_sequences_no_context = create_sequences(X_network, y_categorical, n_time_steps)

# Split data
X_train_no_context, X_test_no_context, y_train_no_context, y_test_no_context, y_train_original_no_context, y_test_original_no_context = train_test_split(
    X_sequences_no_context, y_sequences_no_context, y_original[n_time_steps-1:], test_size=0.2, random_state=42, stratify=y_original[n_time_steps-1:]
)
# Scale the data
scaler_no_context = StandardScaler()
X_train_scaled_no_context = scaler_no_context.fit_transform(X_train_no_context.reshape(-1, X_network.shape[1])).reshape(X_train_no_context.shape)
X_test_scaled_no_context = scaler_no_context.transform(X_test_no_context.reshape(-1, X_network.shape[1])).reshape(X_test_no_context.shape)


# --- 12.2 Build and Train the LSTM Model (No Context-Awareness) ---
model_no_context = Sequential()
model_no_context.add(LSTM(64, activation='relu', input_shape=(n_time_steps, X_network.shape[1]))) # Use X_network.shape[1]
model_no_context.add(Dense(num_classes, activation='softmax'))

optimizer_no_context = tf.keras.optimizers.SGD(learning_rate=0.01)
model_no_context.compile(optimizer=optimizer_no_context, loss='categorical_crossentropy', metrics=['accuracy'])
model_no_context.summary()

history_no_context = model_no_context.fit(X_train_scaled_no_context, y_train_no_context, epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose=1)

# --- 12.3 Evaluate the Model and Get Metrics (No Context-Awareness) ---
y_pred_prob_no_context = model_no_context.predict(X_test_scaled_no_context)
y_pred_classes_no_context = np.argmax(y_pred_prob_no_context, axis=1)
y_pred_original_no_context = label_encoder.inverse_transform(y_pred_classes_no_context)

print("\n--- Evaluation Metrics (No Context-Awareness) ---")
accuracy_no_context = accuracy_score(y_test_original_no_context, y_pred_original_no_context)
precision_no_context = precision_score(y_test_original_no_context, y_pred_original_no_context, average='weighted')
recall_no_context = recall_score(y_test_original_no_context, y_pred_original_no_context, average='weighted')
f1_no_context = f1_score(y_test_original_no_context, y_pred_original_no_context, average='weighted')
loss_no_context = model_no_context.evaluate(X_test_scaled_no_context, y_test_no_context, verbose=0)[0]

print(f"Accuracy: {accuracy_no_context:.4f}")
print(f"Precision (weighted): {precision_no_context:.4f}")
print(f"Recall (weighted): {recall_no_context:.4f}")
print(f"F1-Score (weighted): {f1_no_context:.4f}")
print(f"Loss: {loss_no_context:.4f}")

print("\n--- Classification Report (No Context-Awareness) ---")
print(classification_report(y_test_original_no_context, y_pred_original_no_context))

# --- 13. Plotting Training History (No Context-Awareness) ---
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_no_context.history['accuracy'], label='Train Accuracy')
plt.plot(history_no_context.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy (No Context-Awareness)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_no_context.history['loss'], label='Train Loss')
plt.plot(history_no_context.history['val_loss'], label='Validation Loss')
plt.title('Model Loss (No Context-Awareness)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

# --- 14. Confusion Matrix (No Context-Awareness) ---
conf_matrix_no_context = confusion_matrix(y_test_original_no_context, y_pred_original_no_context)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_no_context, annot=True, fmt='d', cmap='Blues',
            xticklabels=unique_traffic_types, yticklabels=unique_traffic_types)
plt.xlabel('Predicted Traffic Type')
plt.ylabel('Actual Traffic Type')
plt.title('Confusion Matrix (LSTM - No Context-Awareness)')
plt.show()

# --- 15. AUC-ROC Curve (No Context-Awareness) ---
plt.figure(figsize=(10, 8))
lw = 2
y_test_encoded_no_context = label_encoder.transform(y_test_original_no_context)
y_test_onehot_no_context = to_categorical(y_test_encoded_no_context, num_classes=num_classes)


for i, traffic_type in enumerate(unique_traffic_types):
    fpr_no_context, tpr_no_context, _ = roc_curve(y_test_onehot_no_context[:, i], y_pred_prob_no_context[:,